<a href="https://colab.research.google.com/github/safestepassistant/Python/blob/main/SparkTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Code required to get Spark environment

In [ ]:
!pip install pyspark

In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when


spark = SparkSession.builder.appName("LegalDocumentProcessing").getOrCreate()


df = spark.read.csv("data-engineer-interview-data (2).csv", header=True, inferSchema=True)


df.show(5)

+--------------+------------+------------+----------------------+--------------------+------------+
|DocumentNumber|DocumentDate|DocumentType|RefersToDocumentNumber|RefersToDocumentYear|     Remarks|
+--------------+------------+------------+----------------------+--------------------+------------+
|            27|   8/16/2021|           C|                  NULL|                NULL|        NULL|
|             1|   2/15/2021|           T|                     8|                2020|2020 0008-00|
|            67|   10/9/2020|           A|                  NULL|                NULL|        NULL|
|           157|   10/9/2020|           A|                  NULL|                NULL|        NULL|
|           189|   10/9/2020|           J|                  NULL|                NULL|        NULL|
+--------------+------------+------------+----------------------+--------------------+------------+
only showing top 5 rows


In [37]:

ref_docs_fixed = (
    df.filter(col("DocumentType").isin("T", "R"))
      .select(
          col("RefersToDocumentNumber").alias("LinkedDocID"),
          col("DocumentType").alias("ReferencingType")
      )
      .filter(col("LinkedDocID").isNotNull())
      .distinct()
)

In [38]:
ref_docs_fixed.show(5)

+-----------+---------------+
|LinkedDocID|ReferencingType|
+-----------+---------------+
|          1|              R|
|       2013|              T|
|       2013|              R|
|       2015|              R|
|       1995|              T|
+-----------+---------------+
only showing top 5 rows


In [39]:
df_joined_fixed = df.join(
    ref_docs_fixed,
    df["DocumentNumber"] == ref_docs_fixed["LinkedDocID"],
    "left"
)

df_joined_fixed.select("DocumentNumber", "DocumentType", "LinkedDocID", "ReferencingType").show(10)
print("Joined row count:", df_joined_fixed.count())

+--------------+------------+-----------+---------------+
|DocumentNumber|DocumentType|LinkedDocID|ReferencingType|
+--------------+------------+-----------+---------------+
|            27|           C|       NULL|           NULL|
|             1|           T|          1|              R|
|            67|           A|       NULL|           NULL|
|           157|           A|       NULL|           NULL|
|           189|           J|       NULL|           NULL|
|           250|           R|       NULL|           NULL|
|            87|           R|       NULL|           NULL|
|             8|           B|          8|              T|
|             2|           C|       NULL|           NULL|
|            34|           A|       NULL|           NULL|
+--------------+------------+-----------+---------------+
only showing top 10 rows
Joined row count: 146


In [40]:
df_processed = df_joined_fixed.withColumn(
    "ToRemove",
    when((col("DocumentType") == "J") & (col("ReferencingType") == "T"), True)
    .when((col("DocumentType") != "J") & (col("ReferencingType") == "R"), True)
    .otherwise(False)
)

In [41]:

audit_trail = df_processed.filter(col("ToRemove") == True)


final_df = df_processed.filter(col("ToRemove") == False)

In [42]:
original_count = df.count()
audit_count = audit_trail.count()
final_count = final_df.count()

print(f"Original Count: {original_count}")
print(f"Audit Trail (Removed) Count: {audit_count}")
print(f"Final (Remaining) Count: {final_count}")
print(f"Sum (Audit + Final): {audit_count + final_count}")


if (audit_count + final_count) == original_count:
    print("Verification Successful: Counts match.")
else:
    print("Verification Failed: Counts do not match.")

Original Count: 146
Audit Trail (Removed) Count: 4
Final (Remaining) Count: 142
Sum (Audit + Final): 146
Verification Successful: Counts match.
